# 01 — EDA (FathomNet / CLEF 2026)

**Input:** saved output of `rscharan/00-setup-data-download-fathomnetclef2026` (contains `train_dataset.json`, `test_dataset.json`, `images/`), attached via `kernel_sources`.

Research-grade dataset characterization for the paper's Dataset section: integrity audit, class-imbalance quantification, COCO size breakdown, image-level statistics, resolution/aspect-ratio analysis, and sample figures. All summary numbers are exported to `eda_summary.json` + CSV tables.

In [ ]:
# --- 1. Setup, load, integrity audit -------------------------------------
from pathlib import Path
import json, glob
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='colorblind', font_scale=1.1)
mpl.rcParams.update({'figure.dpi': 110, 'savefig.bbox': 'tight'})

OUT = Path('/kaggle/working')

hits = glob.glob('/kaggle/input/**/train_dataset.json', recursive=True)
ANN = Path(hits[0]).parent if hits else Path('/kaggle/working')
IMG = ANN.parent / 'images'
if not IMG.exists():
    IMG = ANN / 'images'

train = json.load(open(ANN / 'train_dataset.json'))
test = json.load(open(ANN / 'test_dataset.json'))

cats = {c['id']: c['name'] for c in train['categories']}
img_by_id = {i['id']: i for i in train['images']}

# integrity audit
missing = [i['file_name'] for i in train['images'] if not (IMG / i['file_name']).exists()]
orphan = [a for a in train['annotations'] if a['image_id'] not in img_by_id]
anns_by_img = {}
for a in train['annotations']:
    anns_by_img.setdefault(a['image_id'], []).append(a)
empty_imgs = [i['id'] for i in train['images'] if i['id'] not in anns_by_img]
bad_boxes = [a for a in train['annotations'] if a['bbox'][2] <= 0 or a['bbox'][3] <= 0]

print(f'train: {len(train["images"])} imgs, {len(train["annotations"])} anns, {len(train["categories"])} cats')
print(f'test : {len(test["images"])} imgs, {len(test.get("annotations", []))} anns')
print(f'audit -> missing image files: {len(missing)}, orphan annotations: {len(orphan)}, '
      f'empty images (0 boxes): {len(empty_imgs)}, degenerate boxes (w or h <= 0): {len(bad_boxes)}')

In [ ]:
# --- 2. Class distribution + imbalance quantification --------------------
df = pd.DataFrame(train['annotations'])
df['name'] = df['category_id'].map(cats)
counts = df['name'].value_counts()

# Gini coefficient of class-frequency concentration
v = counts.values.astype(float)
gini = (2 * np.arange(1, len(v) + 1) * np.sort(v) - (len(v) + 1) * np.sort(v)).sum() / (len(v) * v.sum())
imb_ratio = counts.max() / counts.min()

# head / mid / tail split by instance share
shares = v / v.sum()
cum = np.cumsum(np.sort(shares)[::-1])
n_head = int((cum < 0.50).sum()) + 1          # classes covering first 50% of instances
n_mid = int(((cum >= 0.50) & (cum < 0.80)).sum())
n_tail = len(v) - n_head - n_mid              # classes covering last 20%

fig, ax = plt.subplots(figsize=(8, 10))
counts.plot.barh(ax=ax, color=mpl.cm.Blues(0.25 + 0.65 * (v.max() - v) / max(v.max() - v.min(), 1)))
ax.set_xscale('log'); ax.invert_yaxis()
ax.set(title='Annotation counts per class (train)', xlabel='instances (log scale)', ylabel='')
fig.savefig(OUT / 'fig_class_distribution.pdf'); fig.savefig(OUT / 'fig_class_distribution.png', dpi=200)
plt.show()

print(f'imbalance ratio (max/min): {imb_ratio:.1f}   Gini: {gini:.3f}')
print(f'head/mid/tail classes (50/30/20% of instances): {n_head}/{n_mid}/{n_tail}')
counts_df = counts.rename_axis('class').reset_index(name='instances')
counts_df.to_csv(OUT / 'table_class_counts.csv', index=False)
counts_df.head(35)

In [ ]:
# --- 3. COCO small/medium/large size breakdown ---------------------------
b = np.array(df['bbox'].tolist())
area = b[:, 2] * b[:, 3]
df['size_cat'] = pd.cut(area, [0, 32**2, 96**2, np.inf], labels=['small', 'medium', 'large'])
size_tab = (df.groupby('name', observed=True)['size_cat'].value_counts(normalize=True)
              .unstack().mul(100).round(1).fillna(0))
overall = df['size_cat'].value_counts(normalize=True).mul(100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 9), gridspec_kw={'width_ratios': [1, 2.2]})
overall.plot.bar(ax=axes[0], color=['#4c72b0', '#dd8452', '#55a868'])
axes[0].set(title='Box sizes, overall', ylabel='% of instances'); axes[0].tick_params(axis='x', rotation=0)
sns.heatmap(size_tab, ax=axes[1], cmap='Blues', vmin=0, vmax=100, cbar_kws={'label': '% of instances'})
axes[1].set(title='Box-size mix per class', xlabel='', ylabel='')
fig.tight_layout()
fig.savefig(OUT / 'fig_size_breakdown.pdf'); fig.savefig(OUT / 'fig_size_breakdown.png', dpi=200)
plt.show()

size_tab.to_csv(OUT / 'table_size_per_class.csv')
print(f'overall: {overall.to_dict()}')
size_tab

In [ ]:
# --- 4. Image-level statistics -------------------------------------------
apimg = pd.Series({k: len(x) for k, x in anns_by_img.items()}, name='anns_per_img')
imgs_per_class = pd.Series({c: len({a['image_id'] for a in train['annotations'] if a['category_id'] == cid})
                            for cid, c in cats.items()}).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
apimg.plot.hist(bins=30, ax=axes[0], color='#4c72b0')
axes[0].set(title=f'Annotations per image (mean {apimg.mean():.1f}, median {apimg.median():.0f})',
            xlabel='boxes in image', ylabel='images')
imgs_per_class.iloc[::-1].plot.barh(ax=axes[1], color='#55a868')
axes[1].set_xscale('log'); axes[1].set(title='Images containing each class', xlabel='images (log scale)')
fig.tight_layout()
fig.savefig(OUT / 'fig_image_level.pdf'); fig.savefig(OUT / 'fig_image_level.png', dpi=200)
plt.show()

imgs_per_class.rename_axis('class').reset_index(name='images').to_csv(OUT / 'table_images_per_class.csv', index=False)
print(f'images with 0 boxes: {len(empty_imgs)} | max boxes in one image: {apimg.max()}')
imgs_per_class

In [ ]:
# --- 5. Resolution & aspect ratio, train vs test -------------------------
def geom(imgs):
    w = np.array([i['width'] for i in imgs if i.get('width')])
    h = np.array([i['height'] for i in imgs if i.get('height')])
    return pd.DataFrame({'width': w, 'height': h, 'aspect': w / np.maximum(h, 1)})

gtr, gte = geom(train['images']), geom(test['images'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for g, lbl, c in [(gtr, 'train', '#4c72b0'), (gte, 'test', '#dd8452')]:
    sns.kdeplot(np.log2(g['aspect']), ax=axes[0], label=lbl, color=c, fill=True, alpha=.3)
sns.kdeplot(np.log2(gtr['width'] * gtr['height']), ax=axes[1], label='train', color='#4c72b0', fill=True, alpha=.3)
sns.kdeplot(np.log2(gte['width'] * gte['height']), ax=axes[1], label='test', color='#dd8452', fill=True, alpha=.3)
axes[0].set(title='Aspect ratio (log2)', xlabel='log2(w/h)', ylabel='density'); axes[0].legend()
axes[1].set(title='Image area (log2)', xlabel='log2(pixels)', ylabel='density'); axes[1].legend()
fig.tight_layout()
fig.savefig(OUT / 'fig_geometry.pdf'); fig.savefig(OUT / 'fig_geometry.png', dpi=200)
plt.show()

for lbl, g in [('train', gtr), ('test', gte)]:
    print(f'{lbl}: median {int(g["width"].median())}x{int(g["height"].median())}, '
          f'aspect median {g["aspect"].median():.2f} (IQR {g["aspect"].quantile(.25):.2f}-{g["aspect"].quantile(.75):.2f})')

In [ ]:
# --- 6. Sample images with GT boxes (paper figure) -----------------------
import matplotlib.patches as patches
from PIL import Image

rng = np.random.default_rng(42)   # reproducible sample
have = [i for i in train['images'] if (IMG / i['file_name']).exists()]
pick = rng.choice(have, size=min(8, len(have)), replace=False)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, im in zip(axes.ravel(), pick):
    ax.imshow(Image.open(IMG / im['file_name']))
    for a in anns_by_img.get(im['id'], []):
        x, y, w, h = a['bbox']
        ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, ec='#d62728', lw=1.6))
        ax.text(x, max(y - 4, 8), cats[a['category_id']], color='white', fontsize=7,
                bbox={'fc': '#1f1f1f', 'alpha': .7, 'pad': 1.5})
    ax.set_title(f'{len(anns_by_img.get(im["id"], []))} boxes', fontsize=9)
    ax.axis('off')
fig.suptitle('Randomly sampled train images with ground-truth boxes (seed=42)', y=1.0)
fig.tight_layout()
fig.savefig(OUT / 'fig_samples.pdf'); fig.savefig(OUT / 'fig_samples.png', dpi=200)
plt.show()

In [ ]:
# --- 7. Export summary artifact ------------------------------------------
summary = {
    'train_images': len(train['images']), 'train_annotations': len(train['annotations']),
    'n_classes': len(train['categories']), 'test_images': len(test['images']),
    'imbalance_ratio': round(float(imb_ratio), 1), 'gini': round(float(gini), 3),
    'head_mid_tail': [n_head, n_mid, n_tail],
    'size_overall_pct': overall.to_dict(),
    'anns_per_image': {'mean': round(float(apimg.mean()), 2), 'median': float(apimg.median()), 'max': int(apimg.max())},
    'empty_images': len(empty_imgs), 'missing_files': len(missing), 'orphan_annotations': len(orphan),
    'train_median_size': [int(gtr['width'].median()), int(gtr['height'].median())],
    'train_median_aspect': round(float(gtr['aspect'].median()), 2),
}
json.dump(summary, open(OUT / 'eda_summary.json', 'w'), indent=2)
print(json.dumps(summary, indent=2))